# load_openaire_researchproduct_subjects

Prototipo del nodo `load_openaire_researchproduct_subjects` del pipeline `load_openaire`. No guarda datasets.


In [ ]:
from datetime import date
import pandas as pd

%load_ext kedro.ipython


In [ ]:
df_researchproduct_raw = catalog.load('raw/openaire/researchproduct/parquet/researchproduct_dev')
df_researchproduct_raw.head(2)


In [ ]:
def _add_openaire_extracted_metadata(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in _EXTRACTED_META_COLS:
        if col not in df.columns:
            df[col] = pd.NA
    return df


In [ ]:
def _add_openaire_loaded_metadata(df: pd.DataFrame, load_datetime=None) -> pd.DataFrame:
    df = df.copy()
    if load_datetime is None:
        load_datetime = date.today()
    df["_load_datetime"] = load_datetime
    return df


In [ ]:
def load_openaire_researchproduct_subjects(df: pd.DataFrame)-> pd.DataFrame:
    df = _add_openaire_extracted_metadata(df)

    df_research_subjects = df.loc[:,['id','subjects', *_EXTRACTED_META_COLS]]
    df_research_subjects.dropna(inplace=True)

    df_research_subjects = df_research_subjects.explode('subjects').reset_index(drop=True)

    df_subjects = pd.json_normalize(df_research_subjects['subjects'])
    df_research_subjects = pd.concat(
        [df_research_subjects[['id', *_EXTRACTED_META_COLS]].reset_index(drop=True), df_subjects.reset_index(drop=True)],
        axis=1,
    )

    df_research_subjects = _add_openaire_loaded_metadata(df_research_subjects)

    return df_research_subjects


In [ ]:
df_research_subjects = load_openaire_researchproduct_subjects(df_researchproduct_raw)


In [ ]:
pd.DataFrame([{'dataset': 'df_research_subjects', 'rows': len(df_research_subjects), 'columns': len(df_research_subjects.columns)}])


In [ ]:
df_research_subjects.head(2)
